### 0.1 — Detect Colab environment

In [ ]:
# ── DETECT COLAB ENVIRONMENT ──────────────────────────────────────────────────
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB = {IN_COLAB}")

### 0.2 — Mount Drive & install langdetect

In [ ]:
# ── MOUNT DRIVE & INSTALL LANGDETECT ──────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
else:
    print("Not in Colab — skipping Drive mount.")

!pip install --quiet datasets langdetect

### 0.3 — Imports, seed, device, paths

In [ ]:
# ── IMPORTS, SEED, DEVICE, PATHS ──────────────────────────────────────────────
import re
import html
import os
import json
import time
import warnings
import gzip

import pandas as pd
import numpy as np

from langdetect import detect, DetectorFactory
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
DetectorFactory.seed = RANDOM_SEED

if IN_COLAB:
    BASE_DIR = '/content/drive/MyDrive/nlp-project/business-case-01'
else:
    BASE_DIR = os.getcwd()

N011_OUTPUT_DIR = os.path.join(BASE_DIR, 'data', 'n011_outputs')
DATASET_DISTIL_DIR = os.path.join(N011_OUTPUT_DIR, 'dataset_clean_distilbert')
DATASET_ROBERTA_DIR = os.path.join(N011_OUTPUT_DIR, 'dataset_clean_roberta')
PLOTS_DIR = os.path.join(N011_OUTPUT_DIR, 'plots')
MODELS_DIR = os.path.join(N011_OUTPUT_DIR, 'models')

for d in [N011_OUTPUT_DIR, DATASET_DISTIL_DIR, DATASET_ROBERTA_DIR, PLOTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Paths configured:")
print(f"  N011_OUTPUT_DIR   : {N011_OUTPUT_DIR}")
print(f"  dataset_distilbert: {DATASET_DISTIL_DIR}")
print(f"  dataset_roberta   : {DATASET_ROBERTA_DIR}")
print(f"  plots             : {PLOTS_DIR}")
print(f"  models            : {MODELS_DIR}")

### 1.1 — Stream JSONL from UCSD, filter 33 categories

In [ ]:
# ── STREAM JSONL FROM UCSD, FILTER 33 CATEGORIES ─────────────────────────────
import requests

SELECTED_CATEGORIES = [
    'All_Beauty', 'Amazon_Fashion', 'Appliances', 'Arts_Crafts_and_Sewing',
    'Automotive', 'Baby_Products', 'Beauty_and_Personal_Care', 'Books',
    'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry',
    'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food',
    'Handmade_Products', 'Health_and_Household', 'Health_and_Personal_Care',
    'Home_and_Kitchen', 'Industrial_and_Scientific', 'Kindle_Store',
    'Magazine_Subscriptions', 'Movies_and_TV', 'Musical_Instruments',
    'Office_Products', 'Patio_Lawn_and_Garden', 'Pet_Supplies', 'Software',
    'Sports_and_Outdoors', 'Subscription_Boxes', 'Tools_and_Home_Improvement',
    'Toys_and_Games', 'Unknown',
]
N_SAMPLES_PER_CATEGORY = 30_000
BASE_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories'

all_records = []
for category in SELECTED_CATEGORIES:
    print(f"\n📦 Loading category: {category}")
    url = f'{BASE_URL}/{category}.jsonl.gz'
    try:
        response = requests.get(url, stream=True, timeout=600)
        response.raise_for_status()
        category_records = []
        with gzip.GzipFile(fileobj=response.raw, mode='rb') as gz:
            pbar = tqdm(total=N_SAMPLES_PER_CATEGORY, desc=f'  {category}', leave=False, unit=' reviews')
            for line in gz:
                record = json.loads(line.decode('utf-8'))
                record['category'] = category
                category_records.append(record)
                pbar.update(1)
                if len(category_records) >= N_SAMPLES_PER_CATEGORY:
                    break
            pbar.close()
        all_records.extend(category_records)
        print(f'  ✅ Collected {len(category_records):,} records from {category}')
    except Exception as e:
        print(f'  ⚠️  Skipping {category} — {e}')
        continue

df = pd.DataFrame(all_records)
print(f"\n✅ Raw DataFrame shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
assert len(df) > 100_000, f"Expected >100k rows, got {len(df)}"
print(f"\n📊 Category counts:\n{df['category'].value_counts()}")

### 2.1 — Detect English reviews via langdetect

In [ ]:
# ── DETECT ENGLISH REVIEWS VIA LANGDETECT ─────────────────────────────────────
def is_english(text: str) -> bool:
    try:
        return detect(text[:200]) == 'en'
    except:
        return True  # Too short → assume English (conservative)

df["is_english"] = df["text"].apply(is_english)
pct_non_en = 100 * (~df["is_english"]).sum() / len(df)
print(f"Non-English reviews: {(~df['is_english']).sum():,} ({pct_non_en:.1f}%)")
print(f"Languages found: sample 10 non-English...")
non_en_sample = df[~df["is_english"]].sample(min(10, (~df["is_english"]).sum()), random_state=RANDOM_SEED)
for idx, row in non_en_sample.iterrows():
    print(f"  [{row['category']}] {str(row['text'])[:120]}...")
assert pct_non_en < 20, f"Non-English pct {pct_non_en:.1f}% is unexpectedly high"

### 3.1 — Text cleaning functions (two variants)

In [ ]:
# ── TEXT CLEANING FUNCTIONS (TWO VARIANTS) ────────────────────────────────────
def clean_distilbert(text: str) -> str:
    """Pipeline A: lowercase + remove punct + ASCII clean."""
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_roberta(text: str) -> str:
    """Pipeline B: lowercase + keep punctuation + keep UTF-8."""
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    # KEEP punctuation — RoBERTa BPE handles it
    # KEEP UTF-8 — no ascii encode/decode
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Smoke tests
test_dirty = "  <br>Great &amp; fast! Visit http://spam.com 😊 5★  "
print(f"DistilBERT: '{clean_distilbert(test_dirty)}'")
print(f"RoBERTa   : '{clean_roberta(test_dirty)}'")

### 3.2 — Apply cleaning to English-only reviews

In [ ]:
# ── APPLY CLEANING TO ENGLISH-ONLY REVIEWS ────────────────────────────────────
df_en = df[df["is_english"]].copy()
print(f"English reviews retained: {len(df_en):,} / {len(df):,} ({100*len(df_en)/len(df):.1f}%)")

tqdm.pandas(desc="Cleaning DistilBERT")
df_en["text_distilbert"] = df_en["text"].progress_apply(clean_distilbert)
tqdm.pandas(desc="Cleaning RoBERTa")
df_en["text_roberta"] = df_en["text"].progress_apply(clean_roberta)
print("\n✅ Cleaning complete for both pipelines.")

# Verify before/after on random sample for BOTH pipelines
print("\nBefore/After sample (3 rows) — DistilBERT:")
for idx in df_en.sample(3, random_state=RANDOM_SEED).index:
    print(f"  RAW     : {str(df_en.loc[idx, 'text'])[:100]}")
    print(f"  DISTIL  : {str(df_en.loc[idx, 'text_distilbert'])[:100]}")
print("\nBefore/After sample (3 rows) — RoBERTa:")
for idx in df_en.sample(3, random_state=RANDOM_SEED+1).index:
    print(f"  RAW     : {str(df_en.loc[idx, 'text'])[:100]}")
    print(f"  ROBERTA : {str(df_en.loc[idx, 'text_roberta'])[:100]}")

### 3.3 — Remove duplicates & short reviews

In [ ]:
# ── REMOVE DUPLICATES & SHORT REVIEWS ─────────────────────────────────────────
len_before = len(df_en)

# Drop exact duplicates on original text
df_en = df_en.drop_duplicates(subset=['text']).copy()
# Drop near-duplicates (normalized original text)
norm_text = df_en['text'].str.lower().str.strip()
df_en = df_en[~norm_text.duplicated()].copy()
n_deduped = len_before - len(df_en)
print(f"Duplicates removed: {n_deduped:,}")

# Min length filter — apply to BOTH pipelines
MIN_TEXT_LENGTH = 10
mask_distil = df_en['text_distilbert'].str.len() >= MIN_TEXT_LENGTH
mask_roberta = df_en['text_roberta'].str.len() >= MIN_TEXT_LENGTH
df_en = df_en[mask_distil & mask_roberta].copy()
n_short = len_before - n_deduped - len(df_en)
print(f"Short reviews removed (<{MIN_TEXT_LENGTH} chars in either pipeline): {n_short:,}")
print(f"Final count after dedup + length filter: {len(df_en):,}")

### 4.1 — Derive sentiment labels from rating

In [ ]:
# ── DERIVE SENTIMENT LABELS FROM RATING ───────────────────────────────────────
def rating_to_label(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df_en['label'] = df_en['rating'].astype(int).apply(rating_to_label)
label_counts = df_en['label'].value_counts().sort_index()
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
print("Label distribution:")
for lid, cnt in label_counts.items():
    print(f"  {lid} ({label_names[lid]:8s}): {cnt:,} ({cnt/len(df_en)*100:.1f}%)")

### 4.2 — Stratified split 70/15/15 + Arrow export

In [ ]:
# ── STRATIFIED SPLIT 70/15/15 + ARROW EXPORT ──────────────────────────────────
TEST_RATIO = 0.15
VAL_RATIO = 0.15

# Single stratified split used for BOTH pipelines (fair comparison)
df_trainval, df_test = train_test_split(
    df_en, test_size=TEST_RATIO, stratify=df_en['label'], random_state=RANDOM_SEED
)
val_fraction = VAL_RATIO / (1 - TEST_RATIO)
df_train, df_val = train_test_split(
    df_trainval, test_size=val_fraction, stratify=df_trainval['label'], random_state=RANDOM_SEED
)

# Verify no data leakage
assert len(set(df_train.index) & set(df_val.index)) == 0, "Leakage: train ∩ val"
assert len(set(df_train.index) & set(df_test.index)) == 0, "Leakage: train ∩ test"
assert len(set(df_val.index) & set(df_test.index)) == 0, "Leakage: val ∩ test"
print("✅ No data leakage detected.")

KEEP_COLS = ['text_distilbert', 'text_roberta', 'label', 'rating', 'category']

def save_pipeline(df_tr, df_va, df_te, text_col, save_dir):
    dsd = DatasetDict({
        'train': Dataset.from_pandas(df_tr[KEEP_COLS].rename(columns={text_col: 'text'}), preserve_index=False),
        'validation': Dataset.from_pandas(df_va[KEEP_COLS].rename(columns={text_col: 'text'}), preserve_index=False),
        'test': Dataset.from_pandas(df_te[KEEP_COLS].rename(columns={text_col: 'text'}), preserve_index=False),
    })
    dsd.save_to_disk(save_dir)
    print(f"✅ Saved {text_col} → {save_dir}")
    print(f"   Train: {len(dsd['train']):,} | Val: {len(dsd['validation']):,} | Test: {len(dsd['test']):,}")

save_pipeline(df_train, df_val, df_test, 'text_distilbert', DATASET_DISTIL_DIR)
save_pipeline(df_train, df_val, df_test, 'text_roberta', DATASET_ROBERTA_DIR)

# Final assertions
for name, d in [('distilbert', DATASET_DISTIL_DIR), ('roberta', DATASET_ROBERTA_DIR)]:
    assert os.path.isdir(d), f"{name} dir missing: {d}"
    for split in ['train', 'validation', 'test']:
        assert os.path.isdir(os.path.join(d, split)), f"{name}/{split} missing"
print("\n✅ All Arrow directories validated.")

### 5.1 — Final dataset statistics

In [ ]:
# ── FINAL DATASET STATISTICS ──────────────────────────────────────────────────
final_count = len(df_en)
print("="*60)
print("  N01.1 — DATA PREP COMPLETE")
print("="*60)
print(f"  Raw reviews loaded    : {len(df):,}")
print(f"  Non-English removed   : {(~df['is_english']).sum():,} ({pct_non_en:.1f}%)")
print(f"  English reviews       : {len(df_en):,}")
print(f"  After dedup + length  : {final_count:,}")
print(f"\n  Outputs:")
print(f"    {DATASET_DISTIL_DIR}")
print(f"    {DATASET_ROBERTA_DIR}")
print(f"\n  → Next: N02 uses dataset_clean_distilbert")
print(f"  → Next: N03.1 uses dataset_clean_roberta")